# Exploring Particle Health Flat Data

This notebook walks through Particle Health's **flat JSON format** — a denormalized view of clinical data that's easier to work with than FHIR for common use cases.

**No credentials required.** We use the sample data file included in this repo (`sample-data/flat_data.json`).

Each top-level key is a resource type (e.g., `medications`, `problems`, `labs`), and each value is a list of flat records with all fields as strings.

In [ ]:
import json
import os

# Load sample data
data_path = os.path.join("..", "sample-data", "flat_data.json")
with open(data_path) as f:
    data = json.load(f)

file_size_kb = os.path.getsize(data_path) / 1024
print(f"File size: {file_size_kb:.1f} KB")
print(f"Resource types: {len(data)}")

In [ ]:
# Resource type overview
print(f"{'Resource Type':<25} {'Records':>8}  {'Fields (first record)':>22}")
print("-" * 60)
for key in sorted(data.keys()):
    records = data[key]
    count = len(records)
    fields = len(records[0]) if records else 0
    print(f"{key:<25} {count:>8}  {fields:>22}")

In [ ]:
# Medications
medications = data.get("medications", [])
print(f"Medications: {len(medications)} records\n")

for med in medications:
    name = med.get("medication_name", "Unknown")
    route = med.get("medication_statement_dose_route", "")
    dose = med.get("medication_statement_dose_value", "")
    unit = med.get("medication_statement_dose_unit", "")
    code = med.get("medication_code", "")
    print(f"  {name}")
    if dose and unit:
        print(f"    Dose: {dose} {unit} ({route})")
    if code:
        print(f"    Code: {code}")

In [ ]:
# Problems / Conditions
problems = data.get("problems", [])
print(f"Problems: {len(problems)} records\n")

for prob in problems:
    name = prob.get("condition_name", "Unknown")
    status = prob.get("condition_clinical_status", "")
    onset = prob.get("condition_onset_date", "")
    code = prob.get("condition_code", "")
    print(f"  {name} [{status}]")
    if onset:
        print(f"    Onset: {onset}")
    if code:
        print(f"    Code: {code}")

In [ ]:
# Encounters
encounters = data.get("encounters", [])
print(f"Encounters: {len(encounters)} records\n")

for enc in encounters:
    enc_type = enc.get("encounter_type_name", "Unknown")
    start = enc.get("encounter_start_time", "")
    end = enc.get("encounter_end_time", "")
    print(f"  {enc_type}")
    if start:
        period = start
        if end:
            period += f" to {end}"
        print(f"    Period: {period}")

In [ ]:
# Common transformations

# 1. Group vital signs by type
vitals = data.get("vitalSigns", [])
vitals_by_type = {}
for v in vitals:
    name = v.get("vital_sign_observation_name", "Unknown")
    vitals_by_type.setdefault(name, []).append(v)

print(f"Vital sign types ({len(vitals_by_type)}):")
for name, records in sorted(vitals_by_type.items(), key=lambda x: -len(x[1])):
    print(f"  {name}: {len(records)} observations")

# 2. Filter labs with abnormal interpretation
labs = data.get("labs", [])
abnormal = [
    lab for lab in labs
    if lab.get("lab_interpretation") and lab["lab_interpretation"] != "Normal"
]
print(f"\nAbnormal labs: {len(abnormal)} of {len(labs)}")
for lab in abnormal[:5]:
    name = lab.get("lab_name", "Unknown")
    value = lab.get("lab_value", "")
    interp = lab.get("lab_interpretation", "")
    print(f"  {name}: {value} ({interp})")

# 3. Build a simple clinical timeline
timeline = []
for enc in encounters:
    if enc.get("encounter_start_time"):
        enc_name = enc.get("encounter_type_name", "")
        timeline.append((enc["encounter_start_time"], "Encounter", enc_name))
for prob in problems:
    if prob.get("condition_onset_date"):
        cond_name = prob.get("condition_name", "")
        timeline.append((prob["condition_onset_date"], "Condition", cond_name))

timeline.sort(key=lambda x: x[0])
print(f"\nClinical timeline ({len(timeline)} events):")
for ts, event_type, description in timeline:
    print(f"  {ts}  [{event_type}] {description}")

## Next Steps

- **Run with live data:** Use `workflows/hello_particle.py` to fetch real patient data, then point this notebook at the output file.
- **Load into a database:** Use [particle-flat-observatory](../../particle-flat-observatory/) to generate DDL and load flat data into DuckDB or Snowflake.
- **Explore FHIR format:** For richer clinical semantics, retrieve FHIR bundles via `workflows/retrieve_data.py <patient_id> fhir` (production only).
- **Troubleshooting:** See [docs/troubleshooting.md](../docs/troubleshooting.md) for common sandbox issues.